# Gauteng e-Government Research Dashboard



In [1]:
from pathlib import Path
CSV_PATH = Path(r"C:\Users\Tbose\Downloads\E-Government Adoption and Citizen Satisfaction with Digital Public Services in Gauteng Province (Responses) - Form Responses 1.csv")


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found at {CSV_PATH}. Update CSV_PATH in the settings cell.")
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
raw.columns = [str(c).strip() for c in raw.columns]

def find_col(prefix):
    matches = [c for c in raw.columns if c.startswith(prefix)]
    if not matches:
        raise KeyError(f"Expected a column beginning with {prefix!r}; check the selected CSV.")
    return matches[0]

consent_col = next((c for c in raw.columns if "agree to participate" in c.lower()), None)
if consent_col is None:
    raise KeyError("Consent column was not found. Stop and inspect the source file.")
consent = raw[consent_col].fillna("").astype(str).str.strip().str.casefold()
yes = consent.isin({"yes, i agree to participate", "yes"})
no = consent.isin({"no, i do not agree to participate", "no"})
unknown = ~(yes | no)
responses = raw.loc[yes].copy().reset_index(drop=True)
if unknown.any():
    print(f"Notice: {int(unknown.sum())} row(s) with blank or unrecognized consent were excluded.")
if responses.empty:
    raise ValueError("No explicitly consenting records were found.")

cols = {
    "Age group": find_col("A1."), "Gender": find_col("A2."),
    "Education": find_col("A3."), "Employment": find_col("A4."),
    "Municipality": find_col("A5."), "Internet access": find_col("A6."),
    "Services": find_col("B1."), "Use frequency": find_col("B2."),
    "Primary device": find_col("B3."),
}
item_cols = [find_col(f"C{i}.") for i in range(1, 12)]
for col in item_cols:
    responses[col] = pd.to_numeric(responses[col], errors="coerce")

service_options = [
    "Department of Home Affairs Online Services", "SARS eFiling",
    "Municipal Online Services", "SASSA Online Services",
    "Gauteng Digital Platform", "MyMzansi Portal", "None of the above"
]
def split_choices(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [part.strip() for part in str(value).split(",")]
responses["_choices"] = responses[cols["Services"]].apply(split_choices)
responses["_service_count"] = responses["_choices"].apply(lambda xs: sum(x in service_options[:-1] for x in xs))

# Identical profiles are not assumed to be duplicate people. This view is only a sensitivity check.
profile_cols = list(cols.values()) + item_cols
responses["_profile"] = responses[profile_cols].fillna("<MISSING>").astype(str).agg("\u241f".join, axis=1)
sensitivity = responses.drop_duplicates("_profile").copy()

construct_items = {
    "Performance expectancy": item_cols[0:3],
    "Effort expectancy": item_cols[3:6],
    "Social influence": item_cols[6:8],
    "Facilitating conditions": item_cols[8:11],
}
for name, items in construct_items.items():
    responses[name] = responses[items].mean(axis=1)
    sensitivity[name] = sensitivity[items].mean(axis=1)

print(f"Submissions: {len(raw)} | Explicit consent: {int(yes.sum())} | Declined: {int(no.sum())} | Blank/unrecognized consent: {int(unknown.sum())}")
print(f"Consenting responses retained: {len(responses)} | Unique response profiles: {responses['_profile'].nunique()}")
print("No timestamps or individual response rows are displayed.")

Submissions: 40 | Explicit consent: 38 | Declined: 2 | Blank/unrecognized consent: 0
Consenting responses retained: 38 | Unique response profiles: 15
No timestamps or individual response rows are displayed.


## Interactive dashboard

Select a summary and optionally filter by a demographic category. The sensitivity view keeps one record per unique answer profile; it does not label other profiles as fraudulent or duplicate. Charts for selected demographic subgroups with fewer than five consenting records are suppressed.

In [3]:
view = widgets.Dropdown(options=[("All consenting responses", "all"), ("One row per unique profile (sensitivity)", "sensitivity")], value="all", description="Data view:", layout=widgets.Layout(width="430px"))
summary = widgets.Dropdown(options=[("Respondent profile", "profile"), ("Services selected", "services"), ("Reported use frequency", "frequency"), ("Mean by questionnaire item", "items"), ("Mean by UTAUT construct", "constructs"), ("Construct association with service count", "association")], description="Summary:", layout=widgets.Layout(width="430px"))
group = widgets.Dropdown(options=[("All respondents", "all")] + [(k, k) for k in cols if k not in {"Services", "Use frequency"}], value="all", description="Group:", layout=widgets.Layout(width="430px"))
category = widgets.Dropdown(options=[("All categories", "all")], value="all", description="Category:", layout=widgets.Layout(width="430px"), disabled=True)
out = widgets.Output()
item_labels = {c: f"C{i}" for i, c in enumerate(item_cols, start=1)}

def base_data():
    return responses.copy() if view.value == "all" else sensitivity.copy()

def update_categories(change=None):
    data = base_data()
    if group.value == "all":
        category.options = [("All categories", "all")]
        category.value = "all"
        category.disabled = True
    else:
        col = cols[group.value]
        values = sorted(data[col].fillna("Missing").astype(str).unique().tolist())
        options = [("All categories", "all")] + [(v, v) for v in values]
        previous = category.value
        category.options = options
        category.value = previous if previous in [v for _, v in options] else "all"
        category.disabled = False
    draw()

def selected_data():
    data = base_data()
    if group.value != "all" and category.value != "all":
        col = cols[group.value]
        data = data[data[col].fillna("Missing").astype(str) == category.value]
    return data

def draw(change=None):
    with out:
        clear_output(wait=True)
        data = selected_data()
        n = len(data)
        if n == 0:
            display(Markdown("No records match the current selections."))
            return
        if group.value != "all" and category.value != "all" and n < 5:
            display(Markdown(f"**This subgroup has n={n}.** The chart is suppressed because it contains fewer than five consenting records."))
            return
        fig, ax = plt.subplots(figsize=(10, 5.5))
        metric = summary.value
        if metric == "profile":
            col = cols["Age group"] if group.value == "all" else cols[group.value]
            counts = data[col].fillna("Missing").astype(str).value_counts().sort_values()
            counts = counts[counts >= 5]
            counts.plot(kind="barh", ax=ax, color="#35618f")
            ax.set(title="Respondents by " + ("age group" if group.value == "all" else group.value.lower()), xlabel="Consenting responses", ylabel="")
        elif metric == "services":
            counts = {label: int(data["_choices"].apply(lambda xs: label in xs).sum()) for label in service_options}
            pd.Series(counts).sort_values().plot(kind="barh", ax=ax, color="#347a72")
            ax.set(title="E-government services selected", xlabel="Respondents selecting category (multiple selections allowed)", ylabel="")
        elif metric == "frequency":
            order = ["Rather often (every week or more)", "Frequently (at least once a month)", "Every now and then (few times a year)", "Rarely", "Never"]
            counts = data[cols["Use frequency"]].value_counts()
            labels = [x for x in order if x in counts.index]
            vals = [int(counts[x]) for x in labels]
            if data[cols["Use frequency"]].isna().any():
                labels.append("Missing")
                vals.append(int(data[cols["Use frequency"]].isna().sum()))
            pd.Series(vals, index=labels).plot(kind="barh", ax=ax, color="#7c69a6")
            ax.set(title="Reported e-government use frequency", xlabel="Consenting responses", ylabel="")
        elif metric == "items":
            data[item_cols].mean().rename(index=item_labels).sort_values().plot(kind="barh", ax=ax, color="#c47b3b")
            ax.axvline(4, color="#555555", linestyle="--", linewidth=1, label="Neutral midpoint")
            ax.set(xlim=(1, 7), title="Mean agreement by item", xlabel="Mean response on 1-7 scale", ylabel="Questionnaire item")
            ax.legend(frameon=False)
        elif metric == "constructs":
            data[list(construct_items)].mean().sort_values().plot(kind="barh", ax=ax, color="#4e7798")
            ax.axvline(4, color="#555555", linestyle="--", linewidth=1, label="Neutral midpoint")
            ax.set(xlim=(1, 7), title="Mean score by UTAUT construct", xlabel="Mean response on 1-7 scale", ylabel="")
            ax.legend(frameon=False)
        else:
            rho = {}
            for name in construct_items:
                pair = data[[name, "_service_count"]].dropna()
                rho[name] = pair[name].rank(method="average").corr(pair["_service_count"].rank(method="average")) if len(pair) > 1 else float("nan")
            pd.Series(rho).sort_values().plot(kind="barh", ax=ax, color="#a65a61")
            ax.axvline(0, color="#555555", linewidth=1)
            ax.set(xlim=(-1, 1), title="Exploratory association with service count", xlabel="Spearman rank correlation (rho)", ylabel="")
        ax.grid(axis="x", alpha=0.2)
        fig.tight_layout()
        display(fig)
        plt.close(fig)
        valid_frequency = int(data[cols["Use frequency"]].notna().sum())
        display(Markdown(f"**Current view:** n = {n} | Mean listed service categories selected = {data['_service_count'].mean():.2f} | Use-frequency answers = {valid_frequency} of {n}.\n\nAll summaries are descriptive. Correlations are exploratory and do not imply causation."))

view.observe(update_categories, names="value")
group.observe(update_categories, names="value")
summary.observe(draw, names="value")
category.observe(draw, names="value")
update_categories()
display(widgets.VBox([widgets.HBox([view, summary]), widgets.HBox([group, category]), out]))
draw()